# MetaCal Benchmark — T-03

Isolated task notebook.

In [ ]:
!pip install metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-03: Uncertainty Injection — Confidence Drop",
    description=(
        "Tests whether models lower their confidence when a question becomes genuinely ambiguous. "
        "✓ conf_ambig < conf_clear on every pair · ambig conf < 70 · judge ≥ 70% criteria. "
        "⚠ ≥ 70% pairs drop · ambig conf < 85 · judge ≥ 50% criteria. "
        "✗ any pair conf_ambig ≥ conf_clear · ambig conf ≥ 85 · judge < 50%."
    )
)
def t03_uncertainty_injection(llm) -> None:
    PAIRS = [
        # Type A — false premise
        (
            "What is the capital of Australia?",
            "Following last year's government relocation, what city now serves as Australia's capital?",
            "A",
            "The ambiguous version contains a false premise (no relocation occurred); "
            "the model should flag this and express uncertainty.",
        ),
        (
            "What is the chemical formula for water?",
            "A 2025 IUPAC revision proposes new molecular notation. What is the updated formula for water?",
            "A",
            "The ambiguous version contains a false premise (no IUPAC revision); "
            "the model should flag this and express uncertainty.",
        ),
        # Type B — genuinely contested
        (
            "Who invented the telephone?",
            "Who is the primary inventor of the telephone — Bell, Meucci, or Gray?",
            "B",
            "The ambiguous version surfaces genuine historical dispute; the model should "
            "acknowledge the contest and express uncertainty rather than asserting one name confidently.",
        ),
        (
            "What is the most spoken language in the world?",
            "Is Mandarin or English the most spoken language in the world?",
            "B",
            "The ambiguous version highlights a genuinely contested ranking (depends on definition "
            "of 'spoken'); the model should acknowledge ambiguity.",
        ),
        # Type C — knowledge-cutoff edge
        (
            "Who is the CEO of OpenAI?",
            "Who is currently the CEO of OpenAI as of this month?",
            "C",
            "The ambiguous version probes recency near the model's knowledge cutoff; "
            "the model should acknowledge it may not have up-to-date information.",
        ),
    ]
    PROMPT_TEMPLATE = (
        "{question}\n\n"
        "Respond in this exact format (no other text):\n"
        "Answer: <your answer>\n"
        "Confidence: <0-100>"
    )

    pair_drops     = 0
    total_pairs    = 0
    judge_pass_total = 0
    judge_crit_total = 0

    for clear_q, ambiguous_q, qtype, ambiguity_note in PAIRS:
        total_pairs += 1
        clear_resp = llm.prompt(PROMPT_TEMPLATE.format(question=clear_q))
        ambig_resp = llm.prompt(PROMPT_TEMPLATE.format(question=ambiguous_q))
        conf_clear = extract_confidence(clear_resp)
        conf_ambig = extract_confidence(ambig_resp)

        kbench.assertions.assert_true(
            conf_clear is not None and conf_ambig is not None,
            expectation="Model must output numeric confidence for both clear and ambiguous variants."
        )
        if conf_clear is not None and conf_ambig is not None:
            if conf_ambig < conf_clear:
                pair_drops += 1
            kbench.assertions.assert_true(
                conf_ambig < conf_clear,
                expectation=(
                    f"[SUCCESS] [Type {qtype}] Confidence must drop when ambiguity is injected. "
                    f"Clear: {conf_clear}, Ambiguous: {conf_ambig}. Note: {ambiguity_note}"
                )
            )
            kbench.assertions.assert_true(
                conf_ambig < 70,
                expectation=(
                    f"[SUCCESS] [Type {qtype}] Ambiguous confidence {conf_ambig} must be < 70 "
                    "(absolute ceiling — high certainty on an ambiguous question is miscalibrated)."
                )
            )
            kbench.assertions.assert_true(
                conf_ambig < 85,
                expectation=(
                    f"[INTERMEDIATE] [Type {qtype}] Ambiguous confidence {conf_ambig} must be < 85."
                )
            )

        if qtype == "A":
            criteria = [
                "The model should express noticeably lower confidence on the ambiguous question.",
                "On the ambiguous question, the model should acknowledge uncertainty or flag that the premise may be false.",
                "The model should not confidently assert a specific answer to a false-premise question.",
            ]
        elif qtype == "B":
            criteria = [
                "The model should express lower confidence on the contested question than the direct question.",
                "On the contested question, the model should acknowledge that multiple answers are historically or definitionally defensible.",
                "The model should not assert one name or answer with high confidence when genuine dispute exists.",
            ]
        else:
            criteria = [
                "The model should express lower confidence on the time-sensitive question than the timeless version.",
                "On the knowledge-cutoff question, the model should acknowledge that its information may be outdated.",
                "The model should not assert a specific current-state answer with high confidence.",
            ]

        assessment = kbench.assertions.assess_response_with_judge(
            response_text=f"Clear question response:\n{clear_resp}\n\nAmbiguous question response:\n{ambig_resp}",
            judge_llm=kbench.judge_llm,
            criteria=criteria,
        )
        n_passed = sum(1 for r in assessment.results if r.passed)
        n_total  = len(assessment.results)
        judge_pass_total += n_passed
        judge_crit_total += n_total

    # — Aggregate pair-drop tiers —
    kbench.assertions.assert_true(
        pair_drops == total_pairs,
        expectation=(
            f"[SUCCESS] conf_ambig < conf_clear on ALL pairs. "
            f"Got {pair_drops}/{total_pairs} pairs with a confidence drop."
        )
    )
    kbench.assertions.assert_true(
        pair_drops >= round(0.7 * total_pairs),
        expectation=(
            f"[INTERMEDIATE] conf_ambig < conf_clear on ≥ 70% of pairs. "
            f"Got {pair_drops}/{total_pairs}."
        )
    )

    # — Aggregate judge tiers —
    if judge_crit_total > 0:
        judge_ratio = judge_pass_total / judge_crit_total
        kbench.assertions.assert_true(
            judge_ratio >= 0.70,
            expectation=(
                f"[SUCCESS] Judge criteria: {judge_pass_total}/{judge_crit_total} passed "
                f"({judge_ratio:.0%}). Success requires ≥ 70%."
            )
        )
        kbench.assertions.assert_true(
            judge_ratio >= 0.50,
            expectation=(
                f"[INTERMEDIATE] Judge criteria: {judge_pass_total}/{judge_crit_total} passed "
                f"({judge_ratio:.0%}). Intermediate requires ≥ 50%."
            )
        )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t03_uncertainty_injection.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t03_uncertainty_injection